# Days 4–5 — Code-switch-aware post-processing & end-to-end pipeline
The corrector is **conservative**: only confident, HI-tagged, unprotected tokens are changed, every edit is logged, raw output is kept.
Tune rules on the **validation** set only.

In [ ]:
# --- Setup (run once per Colab session) ---
REPO_URL = "https://github.com/arkalatua/Indic_Switch.git"
import os, sys, subprocess
if "google.colab" in sys.modules:
    if not os.path.exists("/content/Indic_Switch"):
        subprocess.run(["git", "clone", REPO_URL, "/content/Indic_Switch"], check=True)
    %cd /content/Indic_Switch
    !pip install -q -r requirements.txt
    # Optional: persist models/results across sessions
    # from google.colab import drive; drive.mount("/content/drive")
else:
    os.chdir(os.path.abspath(".."))  # running locally from notebooks/
sys.path.insert(0, os.getcwd())

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

In [ ]:
from src.pipeline import IndicSwitch
from src.correction import CorrectionConfig
import os
LID = "models/lid_model" if os.path.exists("models/lid_model/config.json") else "models/ngram_lid.pkl"
for script in ["keep", "roman", "mixed"]:
    p = IndicSwitch(lid=LID, correction=CorrectionConfig(target_script=script))
    for t in ["main kal office nhi ja raha, bohot kaam hai yaar", "मेरी मीटिंग कल है, please Ravi ko call karo",
              "haan haan haan haan haan theek hai"]:
        o = p.process_text(t)
        print(f"[{script}] RAW : {o['raw_transcript']}\n        NORM: {o['normalized_transcript']}")
        print("        TAGS:", list(zip(o["tokens"], o["language_tags"], o["confidences"])))
        print("        EDITS:", [(e["before"], e["after"], e["rule"]) for e in o["edits"]])

## Optional: IndicXlit for native-script rendering

In [ ]:
# !pip install -q ai4bharat-transliteration
# p = IndicSwitch(lid=LID, correction=CorrectionConfig(target_script="native"))
# print(p.process_text("main kal office nahi ja raha")["normalized_transcript"])

## End-to-end: `indic_switch(audio_path)`

In [ ]:
from src.pipeline import indic_switch
out = indic_switch("data/samples/sample.wav")
{k: out[k] for k in ["raw_transcript", "normalized_transcript", "language_tags", "timings", "rtf", "segments"]}

## Stress tests: long audio, silence, noise, switches near segment boundaries; slowest stage

In [ ]:
import numpy as np, soundfile as sf
from src.preprocessing import load_audio
from src.asr import DecodeConfig, WhisperASR
wav, sr = load_audio("data/samples/sample.wav")
cases = {"silence": np.zeros(sr * 5, dtype=np.float32),
         "noisy": wav + 0.02 * np.random.randn(len(wav)).astype(np.float32),
         "long": np.concatenate([wav] * max(1, int(60 / (len(wav) / sr))))}
for vad in [False, True]:
    asr = WhisperASR(DecodeConfig(use_vad=vad))
    p = IndicSwitch(asr=asr, lid=LID)
    for name, x in cases.items():
        sf.write(f"/tmp/{name}.wav", x, sr)
        o = p(f"/tmp/{name}.wav")
        t = o["timings"]
        print(f"vad={vad} {name:8s} RTF {o['rtf']:.3f} slowest={max((k for k in t if k != 'total_s'), key=t.get)} | {o['raw_transcript'][:90]}")